# LCIA of European hydrogen distribution markets

This is the master notebook for selecting prospective Brightway databases, running the complete LCIA workflow, reviewing quality-control tables, and creating plots. Calculation and classification logic lives in `run_analysis.py` and `mapping.py`; this notebook contains no LCIA algorithms. The functional unit is **1 kg of hydrogen, gaseous, low pressure**.

## 1. Imports and display settings

In [ ]:
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidate_dirs = [Path.cwd(), Path.cwd() / 'examples' / 'h2-distribution_LCIA']
ANALYSIS_DIR = next((path.resolve() for path in candidate_dirs if (path / 'config.py').exists()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Run this notebook from its own folder or from the repository root.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import config as cfg
from run_analysis import run_analysis, safe_path_component

pd.set_option('display.max_colwidth', 140)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 15, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10,
    'legend.title_fontsize': 11, 'figure.titlesize': 18,
})

## 2. Choose project and databases

This is the only cell that normally needs editing. Run the notebook with the repository's `brightway25` conda environment. Multiple databases are supported; each receives its own results subfolder. The regional steel comparison is optional because it adds a separate analysis scope and substantial runtime.

In [ ]:
PROJECT = 'ecoinvent-3.12-cutoff'
DATABASES = [
    'ecoinvent-3.12-cutoff_SSP2-NDC-remind_2030_04_full',
]
RUN_REGIONAL_STEEL = False
EXPORT_RESULTS = True
SAVE_PLOTS = True

## 3. Run all calculations

The runner selects exact market datasets, calculates EF 3.1, hydrogen-inclusive premise GWP, and aggregate CED scores, performs process and life-cycle-stage contribution analyses, and enforces all reconciliation checks before returning.

In [ ]:
if not DATABASES:
    raise ValueError('Select at least one database.')
if len(DATABASES) != len(set(DATABASES)):
    raise ValueError('DATABASES contains duplicate names.')

results_by_database = {}
for database_name in DATABASES:
    output_dir = (
        cfg.RESULTS_DIR
        if len(DATABASES) == 1
        else cfg.RESULTS_DIR / safe_path_component(database_name)
    )
    print(f'Running {database_name} ...')
    results_by_database[database_name] = run_analysis(
        project=PROJECT,
        database_name=database_name,
        output_dir=output_dir,
        export=EXPORT_RESULTS,
        run_regional_steel=RUN_REGIONAL_STEEL,
    )
    print(f'Finished {database_name}')

## 4. Selection and quality-control review

Review the exact datasets and both reconciliation layers before interpreting plots. Missing sector markets are scenario results: premise creates them only where modeled sector demand is positive.

In [ ]:
for database_name, result in results_by_database.items():
    print(f'\n{database_name}')
    display(result.table('selection'))
    display(result.table('methods'))
    display(result.table('Stage classification audit'))
    display(result.table('Stage reconciliation'))
    display(result.table('Hotspot reconciliation'))
    if RUN_REGIONAL_STEEL:
        display(result.table('Steel-region reconciliation'))

## 5. Accessible, non-repeating visual encoding

A single registry assigns every categorical label one stable color across every database and figure. Different labels never share a color; the code raises instead of recycling the palette. Hatches, markers, and line styles provide redundant encodings for color-vision deficiencies. Heatmaps use a diverging brown–blue-green scale with a neutral midpoint and printed values.

In [ ]:
def categorical_labels(results):
    labels = set(results.market_order)
    labels.update(results.table('Hotspot process groups')['component'].dropna().astype(str))
    labels.update(results.table('Stage Layer 1')['component'].dropna().astype(str))
    labels.update(results.table('Distribution processes')['process'].dropna().astype(str))
    if 'Steel-region order' in results.tables:
        labels.update(results.table('Steel-region order')['region'].dropna().astype(str))
        labels.update(results.table('Steel-region Layer 1')['component'].dropna().astype(str))
        labels.update(results.table('Steel-region distribution')['process'].dropna().astype(str))
    return labels

all_labels = sorted(set().union(*(categorical_labels(result) for result in results_by_database.values())))
if len(all_labels) > len(cfg.ACCESSIBLE_CATEGORICAL_COLORS):
    raise RuntimeError(
        f'{len(all_labels)} categorical labels require unique colors, but the configured accessible palette '
        f'contains {len(cfg.ACCESSIBLE_CATEGORICAL_COLORS)}. Extend the palette; colors will not be recycled.'
    )
COLOR = dict(zip(all_labels, cfg.ACCESSIBLE_CATEGORICAL_COLORS))
HATCH = {label: cfg.HATCHES[i % len(cfg.HATCHES)] for i, label in enumerate(all_labels)}
MARKER = {label: cfg.MARKERS[i % len(cfg.MARKERS)] for i, label in enumerate(all_labels)}
LINESTYLE = {label: cfg.LINESTYLES[i % len(cfg.LINESTYLES)] for i, label in enumerate(all_labels)}
assert len(set(COLOR.values())) == len(COLOR)
print(f'Assigned {len(COLOR)} unique categorical colors without reuse.')

In [ ]:
def finish_figure(fig, result, plot_name, *, rect=None):
    if rect is None:
        fig.tight_layout()
    else:
        fig.tight_layout(rect=rect)
    if SAVE_PLOTS:
        result.output_dir.mkdir(parents=True, exist_ok=True)
        path = result.output_dir / cfg.PLOT_FILENAMES[plot_name]
        fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
        print(f'Saved {path}')
    plt.show()

def plot_absolute_impacts(result):
    scores = result.table('scores')
    categories = scores['impact category'].drop_duplicates().tolist()
    ncols, nrows = 3, math.ceil(len(categories) / 3)
    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 5.2 * nrows), squeeze=False)
    for ax, category in zip(axes.ravel(), categories):
        subset = scores[scores['impact category'] == category].set_index('market').reindex(result.market_order)
        values = subset['score per kg H2'].to_numpy(dtype=float)
        bars = ax.bar(np.arange(len(result.market_order)), values, color=[COLOR[x] for x in result.market_order])
        for bar, label in zip(bars, result.market_order):
            bar.set_hatch(HATCH[label])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(category, pad=12)
        ax.set_ylabel(f"{subset['unit'].iloc[0]} / kg H2")
        ax.set_xticks(np.arange(len(result.market_order)), result.market_order, rotation=38, ha='right')
        for bar, value in zip(bars, values):
            ax.annotate(f'{value:.2e}', (bar.get_x() + bar.get_width()/2, value),
                        xytext=(4, 10 if value >= 0 else -12), textcoords='offset points',
                        ha='center', fontsize=9, rotation=90)
        nonzero = np.abs(values[np.isfinite(values) & (values != 0)])
        if len(nonzero) and nonzero.max() / nonzero.min() > 1000:
            ax.set_yscale('symlog', linthresh=max(nonzero.min(), 1e-30))
    for ax in axes.ravel()[len(categories):]:
        ax.set_visible(False)
    fig.suptitle(f'Absolute LCIA impacts — {result.database_name}', y=1.002)
    finish_figure(fig, result, 'absolute impacts')

def plot_baseline_heatmap(result):
    comparison = result.table('RER comparison')
    categories = result.table('scores')['impact category'].drop_duplicates().tolist()
    matrix = comparison.pivot(index='impact category', columns='market', values='difference (%)').reindex(
        index=categories, columns=result.sector_order)
    finite = matrix.to_numpy(dtype=float)
    finite = finite[np.isfinite(finite)]
    limit = max(10.0, np.percentile(np.abs(finite), 95)) if len(finite) else 100.0
    fig, ax = plt.subplots(figsize=(max(11, 2 * len(result.sector_order)), max(9, .58 * len(categories))))
    image = ax.imshow(matrix, cmap='BrBG', vmin=-limit, vmax=limit, aspect='auto')
    ax.set_xticks(np.arange(len(result.sector_order)), result.sector_order, rotation=35, ha='right')
    ax.set_yticks(np.arange(len(categories)), categories)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iat[i, j]
            if np.isfinite(value):
                ax.text(j, i, f'{value:+.0f}%', ha='center', va='center', fontsize=9,
                        color='white' if abs(value) > .55 * limit else 'black')
    fig.colorbar(image, ax=ax, label='Difference from generic market, %')
    ax.set_title(f'Sector markets relative to Generic (RER) — {result.database_name}')
    finish_figure(fig, result, 'baseline heatmap')

def plot_ef_spider(result):
    ratios = result.table('EF 3.1 spider ratios').set_index('impact category')
    categories = ratios.index.tolist()
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False)
    xdir, ydir = np.sin(angles), np.cos(angles)
    radial_max = max(1.1, float(ratios[result.sector_order].max().max()) * 1.08)
    fig, ax = plt.subplots(figsize=(16, 16))
    circle_angles = np.linspace(0, 2*np.pi, 361)
    for level in sorted(set(np.linspace(0, radial_max, 5).tolist() + [1.0])):
        if level == 0: continue
        ax.plot(level*np.sin(circle_angles), level*np.cos(circle_angles),
                color='black' if np.isclose(level, 1) else '#bdbdbd',
                linewidth=2.0 if np.isclose(level, 1) else .7,
                linestyle='--' if np.isclose(level, 1) else ':')
        ax.text(0, level, f'{level:.2g}', fontsize=10, color='#555555')
    for xd, yd, label in zip(xdir, ydir, categories):
        ax.plot([0, radial_max*xd], [0, radial_max*yd], color='#d0d0d0', linewidth=.6)
        ha = 'center' if abs(xd) < .15 else ('left' if xd > 0 else 'right')
        va = 'center' if abs(yd) < .15 else ('bottom' if yd > 0 else 'top')
        ax.text(radial_max*1.12*xd, radial_max*1.12*yd, label.replace(': ', ':\n'), ha=ha, va=va, fontsize=10)
    for label in result.sector_order:
        values = ratios[label].to_numpy(dtype=float)
        x, y = values*xdir, values*ydir
        ax.plot(np.r_[x, x[0]], np.r_[y, y[0]], color=COLOR[label], marker=MARKER[label],
                linestyle=LINESTYLE[label], linewidth=1.8, markersize=4, label=label)
    limit = radial_max * 1.28
    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), aspect='equal')
    ax.axis('off')
    ax.set_title(f'EF 3.1 impacts relative to Generic (RER) = 1\n{result.database_name}', pad=38)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1.02), frameon=False)
    finish_figure(fig, result, 'EF spider', rect=(0.05, 0.05, .82, .95))

In [ ]:
def plot_signed_stacks(ax, data, markets, component_column, value_column, xlabel, label_threshold=None):
    y = np.arange(len(markets))
    positive = np.zeros(len(markets))
    negative = np.zeros(len(markets))
    maximum = float(data[value_column].abs().max()) if not data.empty else 0.0
    threshold = .03 * maximum if label_threshold is None and maximum else (label_threshold or np.inf)
    for component in data[component_column].drop_duplicates():
        values = data[data[component_column] == component].groupby('market')[value_column].sum().reindex(
            markets, fill_value=0).to_numpy(dtype=float)
        left = np.where(values >= 0, positive, negative)
        bars = ax.barh(y, values, left=left, color=COLOR[str(component)], edgecolor='white',
                       linewidth=.6, label=component, hatch=HATCH[str(component)])
        for bar, value, start in zip(bars, values, left):
            if abs(value) >= threshold:
                ax.text(start + value/2, bar.get_y() + bar.get_height()/2,
                        f'{value:.1f}%' if value_column.endswith('(%)') else f'{value:.2e}',
                        ha='center', va='center', fontsize=8)
        positive += np.where(values >= 0, values, 0)
        negative += np.where(values < 0, values, 0)
    ax.axvline(0, color='black', linewidth=.8)
    ax.set_yticks(y, markets)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)

def plot_hotspots(result):
    data = result.table('Hotspot process groups')
    components = data['component'].drop_duplicates().tolist()
    ncols, nrows = 2, math.ceil(len(cfg.HOTSPOT_IMPACT_CATEGORY_ORDER) / 2)
    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 5.8*nrows), squeeze=False)
    for ax, category in zip(axes.ravel(), cfg.HOTSPOT_IMPACT_CATEGORY_ORDER):
        subset = data[data['impact category'] == category]
        x = np.arange(len(result.market_order))
        positive, negative = np.zeros(len(x)), np.zeros(len(x))
        totals = subset.groupby('market')['score per kg H2'].first().reindex(result.market_order).to_numpy(float)
        for component in components:
            values = subset[subset['component'] == component].groupby('market')['absolute contribution'].sum().reindex(
                result.market_order, fill_value=0).to_numpy(float)
            if not np.any(values): continue
            bottom = np.where(values >= 0, positive, negative)
            bars = ax.bar(x, values, bottom=bottom, color=COLOR[str(component)], edgecolor='white',
                          linewidth=.6, label=component, hatch=HATCH[str(component)])
            shares = np.divide(100*values, totals, out=np.full_like(values, np.nan), where=totals != 0)
            for bar, value, share, start in zip(bars, values, shares, bottom):
                if np.isfinite(share) and abs(share) >= 5:
                    ax.text(bar.get_x()+bar.get_width()/2, start+value/2, f'{share:.0f}%',
                            ha='center', va='center', fontsize=8)
            positive += np.where(values >= 0, values, 0)
            negative += np.where(values < 0, values, 0)
        ax.axhline(0, color='black', linewidth=.8)
        ax.set_title(category)
        ax.set_ylabel(f"Absolute contribution ({subset['unit'].iloc[0]} / kg H2)")
        ax.set_xticks(x, result.market_order, rotation=38, ha='right')
    for ax in axes.ravel()[len(cfg.HOTSPOT_IMPACT_CATEGORY_ORDER):]: ax.set_visible(False)
    handles = {}
    for ax in axes.ravel():
        for handle, label in zip(*ax.get_legend_handles_labels()): handles.setdefault(label, handle)
    fig.legend(handles.values(), handles.keys(), title='Layer 1 process group',
               bbox_to_anchor=(.995, .5), loc='center left', frameon=False)
    fig.suptitle(f'Selected-impact hotspots — {result.database_name}', y=.995)
    finish_figure(fig, result, 'hotspots', rect=(0, 0, .83, .98))

def plot_stage_layer1(result, table='Stage Layer 1', order=None, plot_name='stage layer 1'):
    data = result.table(table)
    markets = order or result.market_order
    fig, ax = plt.subplots(figsize=(16, max(6, .75*len(markets)+2)))
    plot_signed_stacks(ax, data, markets, 'component', 'share of market (%)',
                       'Contribution to total premise-GWP score (%)', label_threshold=3)
    ax.set_title(f'Production technologies and aggregated distribution — {result.database_name}')
    ax.legend(title='Layer 1 component', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
    finish_figure(fig, result, plot_name, rect=(0, 0, .82, 1))

def plot_distribution_layer2(result, table='Distribution processes', order=None, plot_name='distribution layer 2'):
    data = result.table(table)
    markets = order or result.market_order
    fig, ax = plt.subplots(figsize=(14, max(6, .75*len(markets)+2)))
    unit = data['unit'].iloc[0]
    plot_signed_stacks(ax, data, markets, 'process', 'score', f'Absolute contribution ({unit} / kg H2)')
    ax.set_title(f'Transport, conversion, and reconversion — {result.database_name}')
    ax.legend(title='Distribution process', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
    finish_figure(fig, result, plot_name, rect=(0, 0, .78, 1))

## 6. Create and save plots

In [ ]:
for result in results_by_database.values():
    plot_absolute_impacts(result)
    plot_baseline_heatmap(result)
    plot_ef_spider(result)
    plot_hotspots(result)
    plot_stage_layer1(result)
    plot_distribution_layer2(result)
    if RUN_REGIONAL_STEEL:
        steel_order = result.table('Steel-region order')['region'].tolist()
        plot_stage_layer1(result, 'Steel-region Layer 1', steel_order, 'steel stage layer 1')
        plot_distribution_layer2(result, 'Steel-region distribution', steel_order, 'steel distribution layer 2')

## 7. Numerical results and exports

All CSV files are written by `run_analysis.py`. Plot files are written by this notebook. The paths below are the complete handoff for each selected database.

In [ ]:
for database_name, result in results_by_database.items():
    print(f'\n{database_name}')
    if EXPORT_RESULTS:
        display(pd.DataFrame([{'table': label, 'path': str(path)} for label, path in result.export_paths.items()]))
    display(result.table('scores').sort_values(['impact category', 'market']))
    display(result.table('RER comparison'))
    display(result.table('Hotspot process groups'))
    display(result.table('Stage Layer 1'))
    display(result.table('Production input groups'))
    display(result.table('Distribution processes'))

## Interpretation checklist

1. Confirm the exact selection table, especially the generic `RER` versus sector-specific IAM geography.
2. Compare scores only within the same impact category and unit.
3. Treat missing sector markets as a scenario outcome, not an analysis failure.
4. Retain negative contributions and shares above 100%; these can represent credits and offsetting burdens.
5. Use the premise GWP result when hydrogen leakage characterization is required.
6. Read spider values as ratios to Generic (RER), not as EF-normalized or weighted scores.
7. Do not interpret any contribution plot unless its reconciliation table passes.